# Terraform Kubernetes provider

Parses docs from kubernetes provider for terraform

Note: leaves out the "guides" folder, that one is conceptually different from other documents

In [49]:
import markdown
import frontmatter
import re
import pandas as pd
from bs4 import BeautifulSoup, NavigableString
from os import path, makedirs, environ, listdir
from IPython.display import display
from IPython.core.display import HTML
from copy import deepcopy

assets_abspath = path.join(
    environ.get("ASSETS_ABSPATH", ""), "terraform/providers/kubernetes"
)
artifacts_abspath = path.join(
    environ.get("ARTIFACTS_ABSPATH", ""), "terraform/providers/kubernetes"
)
# target_csv_abspath = path.join(artifacts_abspath, "data_sources.csv")
makedirs(artifacts_abspath, exist_ok=True)

In [16]:
types = listdir(assets_abspath)
types = [
    d
    for d in types
    if path.isdir(path.join(assets_abspath, d)) and d != "guides"
]
entries = []
for t in types:
    type_abspath = path.join(assets_abspath, t)
    files = listdir(type_abspath)
    for f in files:
        entries.append(
            {
                "type": t,
                "key": f.replace(".md", ""),
                "filename": f,
                "file_abspath": path.join(type_abspath, f),
            }
        )

In [37]:
ROOT_URL = (
    "https://registry.terraform.io/providers/hashicorp/kubernetes/latest/docs"
)
mds = []
for entry in entries:
    md = open(entry["file_abspath"], "r").read()
    front = frontmatter.loads(md)
    html = markdown.markdown(
        front.content, extensions=["markdown.extensions.fenced_code"]
    )
    soup = BeautifulSoup(html, "html.parser")
    # for anchor in soup.find_all("a"):
    #     try:
    #         if anchor["href"].startswith("/"):
    #             anchor["href"] = "/".join(
    #                 ROOT_URL + entry["type"] + anchor["href"][1:]
    #             )
    #     except KeyError as e:
    #         print(entry["key"], e)
    mds.append(
        {**entry, "soup": soup, "html": html, "frontmatter": front, "md": md}
    )
print(len(mds))

109


In [137]:
enriched = []
for md in mds:
    header_soup = deepcopy(md["soup"].find("h1"))
    header_soup.name = "p"
    header_text = header_soup.text

    content_soup = md["soup"]

    description_text = md["frontmatter"]["description"]
    description_text = description_text.replace(header_text, "___")

    type_reformat = md["type"].replace("-", " ").title().replace(" ", "")
    type_lodash = md["type"].replace("-", "_").lower()

    enriched.append(
        {
            "type": md["type"],
            "type_reformat": type_reformat,
            "header_text": header_text,
            # "header_soup": header_soup,
            # "content_soup": content_soup,
            # "soup": md["soup"],
            # "description_text": description_text,
            "header_html": BeautifulSoup(
                f"<p>{type_lodash}</p>", header_soup.prettify()
            ).prettify(),
            "content_html": content_soup.prettify(),
            "summary_html": f"<p>{description_text}</p>",
            "tags": f"WebScraped {type_reformat} Terraform-Providers-Kubernetes-2.31.0 2024-07-16",
        }
    )

for item in enriched[0:1]:
    display(HTML(item["header_html"]))
    display(HTML(item["summary_html"]))
    # display(HTML(item["content_html"]))
    print("-" * 80)

--------------------------------------------------------------------------------


In [167]:
df = pd.DataFrame(enriched)
df.rename(
    columns={
        "header_html": "HeaderHtml",
        "content_html": "ContentHtml",
        "summary_html": "SummaryHtml",
        "tags": "Tags",
    },
    inplace=True,
)
df

,type,type_reformat,header_text,HeaderHtml,ContentHtml,SummaryHtml,Tags
0,resources,Resources,kubernetes_service,<p>\n kubernetes_service\n</p>\n,<h1>\n kubernetes_service\n</h1>\n<p>\n A Serv...,<p>A Service is an abstraction which defines a...,WebScraped Resources Terraform-Providers-Kuber...
1,resources,Resources,kubernetes_daemon_set_v1,<p>\n kubernetes_daemon_set_v1\n</p>\n,<h1>\n kubernetes_daemon_set_v1\n</h1>\n<p>\n ...,<p>A DaemonSet ensures that all (or some) Node...,WebScraped Resources Terraform-Providers-Kuber...
2,resources,Resources,kubernetes_cron_job,<p>\n kubernetes_cron_job\n</p>\n,<h1>\n kubernetes_cron_job\n</h1>\n<p>\n A Cro...,<p>A Cron Job creates Jobs on a time-based sch...,WebScraped Resources Terraform-Providers-Kuber...
3,resources,Resources,kubernetes_priority_class,<p>\n kubernetes_priority_class\n</p>\n,<h1>\n kubernetes_priority_class\n</h1>\n<p>\n...,<p>A PriorityClass is a non-namespaced object ...,WebScraped Resources Terraform-Providers-Kuber...
4,resources,Resources,kubernetes_mutating_webhook_configuration,<p>\n kubernetes_mutating_webhook_configuratio...,<h1>\n kubernetes_mutating_webhook_configurati...,<p>Mutating Webhook Configuration configures a...,WebScraped Resources Terraform-Providers-Kuber...
...,...,...,...,...,...,...,...
104,data-sources,DataSources,kubernetes_ingress_v1,<p>\n kubernetes_ingress_v1\n</p>\n,<h1>\n kubernetes_ingress_v1\n</h1>\n<p>\n Ing...,<p>Ingress is a collection of rules that allow...,WebScraped DataSources Terraform-Providers-Kub...
105,data-sources,DataSources,kubernetes_config_map,<p>\n kubernetes_config_map\n</p>\n,<h1>\n kubernetes_config_map\n</h1>\n<p>\n Con...,<p>This data source reads configuration data f...,WebScraped DataSources Terraform-Providers-Kub...
106,functions,Functions,function: manifest_decode_multi,<p>\n function: manifest_decode_multi\n</p>\n,<h1>\n function: manifest_decode_multi\n</h1>\...,<p>Decode a Kubernetes YAML manifest containin...,WebScraped Functions Terraform-Providers-Kuber...
107,functions,Functions,function: manifest_encode,<p>\n function: manifest_encode\n</p>\n,<h1>\n function: manifest_encode\n</h1>\n<p>\n...,<p>Decode a Kubernetes YAML manifest containin...,WebScraped Functions Terraform-Providers-Kuber...


In [168]:
for t in types:
    print(t)
    print(df[df["type"] == t].nunique())
    print("-" * 20)

resources
type              1
type_reformat     1
header_text      81
HeaderHtml       81
ContentHtml      81
SummaryHtml      44
Tags              1
dtype: int64
--------------------
data-sources
type              1
type_reformat     1
header_text      25
HeaderHtml       25
ContentHtml      25
SummaryHtml      16
Tags              1
dtype: int64
--------------------
functions
type             1
type_reformat    1
header_text      3
HeaderHtml       3
ContentHtml      3
SummaryHtml      2
Tags             1
dtype: int64
--------------------


In [169]:
# df = df.copy()
for t in types:
    resources = df[df["type"] == t]
    dup = resources[resources.duplicated(subset="SummaryHtml")].copy()
    summary_col_index = df.columns.get_loc("SummaryHtml")
    duplicated_indices = []

    for di, row in dup.iterrows():
        val = row["SummaryHtml"]
        header_suffix = row["header_text"].rsplit("_", 1)[0]
        matches = resources[resources["SummaryHtml"] == val]

        header_texts = []
        for mi, match_row in matches.iterrows():
            duplicated_indices.append(mi)
            header_texts.append({"mi": mi, "text": match_row["header_text"]})
        header_texts.sort(key=lambda a: a["text"])

        default_text = header_texts[0]["text"]
        for c in header_texts:
            c["text"] = c["text"].replace(default_text, "")
            if c["text"] == "":
                c["text"] = "default"
            else:
                c["text"] = c["text"][1:]
        for alterations in header_texts:
            prev = df.iloc[alterations["mi"], summary_col_index]
            replaced = prev.replace("</p>", f" ({alterations['text']})</p>")
            df.iloc[alterations["mi"], summary_col_index] = replaced

df.nunique()

type               3
type_reformat      3
header_text       88
HeaderHtml        88
ContentHtml      109
SummaryHtml       97
Tags               3
dtype: int64

In [170]:
for t in types:
    print(t)
    print(df[df["type"] == t].nunique())
    print("-" * 20)

resources
type              1
type_reformat     1
header_text      81
HeaderHtml       81
ContentHtml      81
SummaryHtml      81
Tags              1
dtype: int64
--------------------
data-sources
type              1
type_reformat     1
header_text      25
HeaderHtml       25
ContentHtml      25
SummaryHtml      25
Tags              1
dtype: int64
--------------------
functions
type             1
type_reformat    1
header_text      3
HeaderHtml       3
ContentHtml      3
SummaryHtml      3
Tags             1
dtype: int64
--------------------


In [171]:
for t in types:
    df[df["type"] == t][
        ["HeaderHtml", "ContentHtml", "SummaryHtml", "Tags"]
    ].to_csv(
        path.join(artifacts_abspath, f"{t}.csv"),
        sep="|",
        index=False,
        header=False,
        encoding="utf-8",
    )